# Mejoras para thyroid


Proyecto de clasificación médica. La mejora clave es no optimizar accuracy, sino detectar bien las clases enfermas.

La métrica importante es F2 porque da más peso al recall.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, fbeta_score, make_scorer
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_PATH = Path("datasets/thyroid/thyroidDF.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("../proyects-upgrades/datasets/thyroid/thyroidDF.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError("Falta thyroidDF.csv. Descárgalo antes de cortar internet.")

df = pd.read_csv(DATA_PATH)


In [ ]:
def simplify_thyroid_class(value: str) -> str:
    if any(code in value for code in ["A", "B", "C", "D"]):
        return "hyperthyroid"
    if any(code in value for code in ["E", "F", "G", "H"]):
        return "hypothyroid"
    return "negative"


df["target"] = df["target"].apply(simplify_thyroid_class)
df = df.dropna(subset=["target"])

X = df.drop(columns=["patient_id", "referral_source", "target"])
y = df["target"]

# Edades imposibles como NaN: no se eliminan filas enteras.
X.loc[X["age"] > 150, "age"] = np.nan

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(y_train.value_counts(normalize=True))


In [ ]:
def thyroid_f2(y_true, y_pred):
    return fbeta_score(
        y_true,
        y_pred,
        beta=2,
        labels=["hyperthyroid", "hypothyroid"],
        average="macro",
        zero_division=0,
    )

thyroid_scorer = make_scorer(thyroid_f2)


In [ ]:
numeric_cols = X_train.select_dtypes(include="number").columns
categorical_cols = X_train.select_dtypes(exclude="number").columns

preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_cols),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore"))]), categorical_cols),
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42, class_weight="balanced", n_jobs=-1)),
])

param_grid = {
    "classifier__n_estimators": [200, 400],
    "classifier__max_depth": [8, 16, None],
    "classifier__min_samples_leaf": [1, 3, 5],
}

search = RandomizedSearchCV(
    model,
    param_distributions=param_grid,
    n_iter=8,
    scoring=thyroid_scorer,
    cv=5,
    random_state=42,
    n_jobs=-1,
)

search.fit(X_train, y_train)
print("Mejores parámetros:", search.best_params_)
print("Mejor F2 CV:", search.best_score_)


In [ ]:
y_pred = search.best_estimator_.predict(X_test)
print(classification_report(y_test, y_pred, zero_division=0))
print("Thyroid F2:", thyroid_f2(y_test, y_pred))


Explicación corta:

“Como es un problema médico y desbalanceado, accuracy no basta. Optimizo F2 sobre las clases enfermas para reducir falsos negativos. Trato edades imposibles como faltantes y uso pesos de clase para no ignorar hiper/hipotiroidismo.”
